# Day Ahead Energy Price Forecast

In [1]:
test ob was ändert

SyntaxError: invalid syntax (3200622417.py, line 1)

In [ ]:
pip install -r requirements.txt

#import sys
#!{sys.executable} -m pip install --force-reinstall pyarrow==15.0.2

#pip install -U "tensorflow[and-cuda]==2.21.0"

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import holidays
import matplotlib.pyplot as plt
from sklearn.preprocessing import RobustScaler
from sklearn.feature_selection import mutual_info_regression

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

import tensorflow as tf
from tensorflow.keras import layers, callbacks
from xgboost import XGBRegressor
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV

import joblib
import pyarrow

I0000 00:00:1774520780.371261   44588 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1774520780.448125   44588 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1774520782.379737   44588 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [ ]:
import os
print(os.environ["LD_LIBRARY_PATH"])


/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/nvidia/cuda_nvrtc/lib:/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/nvidia/cufft/lib:/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/nvidia/cudnn/lib:/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/nvidia/cusparse/lib:/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/nvidia/cublas/lib:/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/nvidia/cufile/lib:/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/nvidia/cuda_cupti/lib:/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/nvidia/nvtx/lib:/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/nvidia/nccl/lib:/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/nvidia/curand/lib:/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/nvidia/cusparselt/lib:/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site

# Load Data

In [ ]:
# ENTSO-E day ahead price and load forecast data
df = pd.read_parquet("ENTSOE/entsoe_price_and_loadfc.parquet")

# for EDA change timezone to Berlin
df.index = df.index.tz_convert("Europe/Berlin")

# filter for 2023-2025
df = df.loc["2023":"2025"]

In [ ]:
# Load previously saved generation forecast parquet files and merge into df
gen_fc_dir = Path("generation_forecast")

gen_fc_wind_on = pd.read_parquet(gen_fc_dir / "wind_on_generation_forecast.parquet").rename(
    columns={"generation_forecast": "gen_fc_wind_on"}
 )
gen_fc_wind_off = pd.read_parquet(gen_fc_dir / "wind_off_generation_forecast.parquet").rename(
    columns={"generation_forecast": "gen_fc_wind_off"}
 )
gen_fc_solar = pd.read_parquet(gen_fc_dir / "solar_generation_forecast.parquet").rename(
    columns={"generation_forecast": "gen_fc_solar"}
 )

# Combine and align timezone to df index
gen_fc = pd.concat([gen_fc_wind_on, gen_fc_wind_off, gen_fc_solar], axis=1).sort_index()
gen_fc = gen_fc.tz_convert("Europe/Berlin")

# Join to existing dataframe
df = df.join(gen_fc, how="left")


# Data Integrity & Time Index Quality

In [ ]:
# Ensure expected period
expected_index = pd.date_range(start="2023-01-01 00:00:00+00:00", end="2025-12-31 23:00:00+00:00", freq="h")
missing_timestamps = expected_index.difference(df.index)
extra_timestamps = df.index.difference(expected_index)

quality_checks = {
    "rows": len(df),
    "expected_rows": len(expected_index),
    "is_monotonic_increasing": bool(df.index.is_monotonic_increasing),
    "has_duplicate_index": bool(df.index.duplicated().any()),
    "missing_hours": int(len(missing_timestamps)),
    "extra_hours": int(len(extra_timestamps)),
    "timezone": str(df.index.tz),
}

quality_df = pd.DataFrame.from_dict(quality_checks, orient="index", columns=["value"])
quality_df

,value
rows,26304
expected_rows,26304
is_monotonic_increasing,True
has_duplicate_index,False
missing_hours,1
extra_hours,1
timezone,Europe/Berlin


In [ ]:
nan_count = df.isna().sum().rename("nan_count")
has_missing = df.isna().any().rename("has_missing")

quality_features = pd.concat([nan_count, has_missing], axis=1)
quality_features


,nan_count,has_missing
da_price,0,False
load_fc,0,False
gen_fc_wind_on,11688,True
gen_fc_wind_off,11688,True
gen_fc_solar,11688,True


First Figure: Some strong outliers that could have a too big impact on the model. However, these are not errors, so these extremes should also be described by the model.

Second Figure: Day ahead price is almost normally distributed but slightly left skewed.

Third Figure: Yearly seasonality visible. Probably month information important for model.

Forth Figure: Strong daily pattern. "Hour of day" variable useful for the model.

Fifth Figure: Strong weekly pattern. "Day of the week" variable useful for the model.

# Add features

# ____________________ FÜR GOOGLE COLAB ____________________

In [ ]:
data = pd.read_parquet("entsoe_price_and_loadfc.parquet")
gen_fc_wind_on = pd.read_parquet("wind_on_generation_forecast.parquet").rename(
    columns={"generation_forecast": "gen_fc_wind_on"}
 )
gen_fc_wind_off = pd.read_parquet("wind_off_generation_forecast.parquet").rename(
    columns={"generation_forecast": "gen_fc_wind_off"}
 )
gen_fc_solar = pd.read_parquet("solar_generation_forecast.parquet").rename(
    columns={"generation_forecast": "gen_fc_solar"}
 )

gen_fc = pd.concat([gen_fc_wind_on, gen_fc_wind_off, gen_fc_solar], axis=1).sort_index()

data = data.join(gen_fc, how="left")

# ____________________ FÜR GOOGLE COLAB ____________________

In [ ]:
# load data again
data = pd.read_parquet("ENTSOE/entsoe_price_and_loadfc.parquet")
#gen_fc = gen_fc.tz_convert("UTC")

#data = data.join(gen_fc, how="left")

In [ ]:
# lag features

## convert to local time to handle daylight saving time correctly
data = data.tz_convert("Europe/Berlin")

idx_naive = data.index.tz_localize(None)

prev_naive_day = idx_naive - pd.DateOffset(days = 1)
prev_naive_day2 = idx_naive - pd.DateOffset(days = 2)
prev_naive_week = idx_naive - pd.DateOffset(weeks = 1)

idx_prev_day = prev_naive_day.tz_localize(
    "Europe/Berlin",
    ambiguous= False,
    nonexistent="shift_forward",
)
idx_prev_day2 = prev_naive_day2.tz_localize(
    "Europe/Berlin",
    ambiguous= False,
    nonexistent="shift_forward",
)
idx_prev_week = prev_naive_week.tz_localize(
    "Europe/Berlin",
    ambiguous= False,
    nonexistent="shift_forward",
)


data["price_lag_24h"] = data["da_price"].reindex(idx_prev_day).to_numpy()
data["price_lag_48h"] = data["da_price"].reindex(idx_prev_day2).to_numpy()
data["price_lag_168h"] = data["da_price"].reindex(idx_prev_week).to_numpy()


# weekday
data["weekday_sin"] = np.sin(2 * np.pi * data.index.dayofweek / 7)  # cyclic, so that distances are correct (Sunday is close to Monday), same for hour
data["weekday_cos"] = np.cos(2 * np.pi * data.index.dayofweek / 7)

# hour of day
data["hour_sin"] = np.sin(2 * np.pi * data.index.hour / 24)
data["hour_cos"] = np.cos(2 * np.pi * data.index.hour / 24)

# holiday not sunday
ger_holidays = list(holidays.Germany(years=data.index.year.unique()))
ger_holidays = pd.to_datetime(ger_holidays).tz_localize(data.index.tz)
data["holiday_not_sunday"] = (data.index.normalize().isin(ger_holidays) & (data.index.dayofweek != 6))


## back to utc
data = data.tz_convert("UTC")

In [ ]:
# remove all data outside of 2023-2025 (only used for lag features)
data = data["2023":"2025"]

In [ ]:
feature_cols = [
    "load_fc",
    "gen_fc_wind_on",
    "gen_fc_wind_off",
    "gen_fc_solar",
    "price_lag_24h",
    "price_lag_48h",
    "price_lag_168h",
    "weekday_sin",
    "weekday_cos",
    "hour_sin",
    "hour_cos",
    "holiday_not_sunday",
]

feature_eval = pd.DataFrame(index=feature_cols)
feature_eval["missing_ratio"] = data[feature_cols].isna().mean()
feature_eval["corr_with_price"] = [data["da_price"].corr(data[c]) for c in feature_cols]

clean_for_mi = data[["da_price"] + feature_cols].copy()
X_mi = clean_for_mi[feature_cols].copy()
X_mi["holiday_not_sunday"] = X_mi["holiday_not_sunday"].astype(int)
mi = mutual_info_regression(X_mi, clean_for_mi["da_price"], random_state=42)
feature_eval["mutual_info"] = mi

feature_eval.loc["hour_combined", "missing_ratio"] = feature_eval.loc[["hour_sin", "hour_cos"], "missing_ratio"].max()
feature_eval.loc["hour_combined", "corr_with_price"] = np.sqrt(
    feature_eval.loc["hour_sin", "corr_with_price"] ** 2 + feature_eval.loc["hour_cos", "corr_with_price"] ** 2
)
feature_eval.loc["hour_combined", "mutual_info"] = np.nan

feature_eval.loc["weekday_combined", "missing_ratio"] = feature_eval.loc[["weekday_sin", "weekday_cos"], "missing_ratio"].max()
feature_eval.loc["weekday_combined", "corr_with_price"] = np.sqrt(
    feature_eval.loc["weekday_sin", "corr_with_price"] ** 2 + feature_eval.loc["weekday_cos", "corr_with_price"] ** 2
)
feature_eval.loc["weekday_combined", "mutual_info"] = np.nan

for y in [2023, 2024, 2025]:
    y_mask = data.index.year == y
    year_corr = pd.Series({
        c: data.loc[y_mask, "da_price"].corr(data.loc[y_mask, c]) for c in feature_cols
    })
    feature_eval[f"corr_{y}"] = year_corr
    feature_eval.loc["hour_combined", f"corr_{y}"] = np.sqrt(
        feature_eval.loc["hour_sin", f"corr_{y}"] ** 2 + feature_eval.loc["hour_cos", f"corr_{y}"] ** 2
    )
    feature_eval.loc["weekday_combined", f"corr_{y}"] = np.sqrt(
        feature_eval.loc["weekday_sin", f"corr_{y}"] ** 2 + feature_eval.loc["weekday_cos", f"corr_{y}"] ** 2
    )


corr_cols = ["corr_2023", "corr_2024", "corr_2025"]
feature_eval["corr_std_across_years"] = feature_eval[corr_cols].std(axis=1)
feature_eval


,missing_ratio,corr_with_price,mutual_info,corr_2023,corr_2024,corr_2025,corr_std_across_years
load_fc,0.0000,0.2841,0.1229,0.3633,0.2335,0.2686,0.0671
gen_fc_wind_on,0.0000,-0.4107,0.1539,-0.4883,-0.3827,-0.3998,0.0567
gen_fc_wind_off,0.0000,-0.2920,0.0879,-0.3445,-0.3095,-0.2223,0.0630
gen_fc_solar,0.0000,-0.2239,0.0971,-0.1422,-0.1756,-0.3438,0.1080
price_lag_24h,0.0000,0.6699,0.4256,0.6418,0.6475,0.6988,0.0314
price_lag_48h,0.0000,0.4872,0.2399,0.4566,0.4281,0.5464,0.0617
price_lag_168h,0.0000,0.5075,0.2718,0.4832,0.4562,0.5518,0.0493
weekday_sin,0.0000,0.2087,0.0443,0.2240,0.2217,0.1885,0.0199
weekday_cos,0.0000,-0.0557,0.0101,-0.0687,-0.0664,-0.0337,0.0196
hour_sin,0.0000,-0.1069,0.0698,-0.1152,-0.1171,-0.0920,0.0140


In [ ]:
# Initial feature whitelist based on low missingness and stable yearly behavior
feature_whitelist = feature_eval[(feature_eval["missing_ratio"] <= 0.02) & (feature_eval["corr_std_across_years"] <= 0.20) & (feature_eval["mutual_info"].notna())].index.tolist()

feature_whitelist

['load_fc',
 'gen_fc_wind_on',
 'gen_fc_wind_off',
 'gen_fc_solar',
 'price_lag_24h',
 'price_lag_48h',
 'price_lag_168h',
 'weekday_sin',
 'weekday_cos',
 'hour_sin',
 'hour_cos',
 'holiday_not_sunday']

# Split data

In [ ]:
TARGET = "da_price"
TRAIN_START = pd.Timestamp("2023-01-01", tz="UTC")
TRAIN_END = pd.Timestamp("2025-01-01", tz="UTC")
TEST_START = pd.Timestamp("2025-01-01", tz="UTC")
TEST_END = pd.Timestamp("2026-01-01", tz="UTC")

train_mask = (data.index >= TRAIN_START) & (data.index < TRAIN_END)
test_mask = (data.index >= TEST_START) & (data.index < TEST_END)

data_train = data.loc[train_mask].copy()
data_test = data.loc[test_mask].copy()


split_summary = pd.DataFrame({
    "split": ["train", "test"],
    "start": [data_train.index.min(), data_test.index.min()],
    "end": [data_train.index.max(), data_test.index.max()],
    "rows": [len(data_train), len(data_test)],
})
split_summary

,split,start,end,rows
0,train,2023-01-01 00:00:00+00:00,2024-12-31 23:00:00+00:00,17544
1,test,2025-01-01 00:00:00+00:00,2025-12-31 23:00:00+00:00,8760


# Baseline model

### Error Functions

In [ ]:
def mae_score(y_true, y_pred) -> float:
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return float(np.mean(np.abs(y_true - y_pred)))


def rmse_score(y_true, y_pred) -> float:
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

In [ ]:
y_test = data_test[TARGET]
pred_daily = data_test["price_lag_24h"]
pred_weekly = data_test["price_lag_168h"]

baseline_scores = pd.DataFrame(
    [
        {"baseline": "daily_persistence_lag24", "mae": mae_score(y_test, pred_daily), "rmse": rmse_score(y_test, pred_daily)},
        {"baseline": "weekly_persistence_lag168", "mae": mae_score(y_test, pred_weekly), "rmse": rmse_score(y_test, pred_weekly)},
    ]
).sort_values("mae")

baseline_scores

,baseline,mae,rmse
0,daily_persistence_lag24,25.9650,40.4489
1,weekly_persistence_lag168,32.8043,49.3715


# LSTM-Readiness Checks

### General Setup

In [ ]:
# Compact setup vars
LOOKBACK = 168
HORIZON_STEPS = 1
TARGET_COL = TARGET

FEATURE_COLS = ['load_fc',
                'gen_fc_wind_on',
                'gen_fc_wind_off',
                'gen_fc_solar',
                'price_lag_24h',
                'price_lag_48h',
                'price_lag_168h',
                'weekday_sin',
                'weekday_cos',
                'hour_sin',
                'hour_cos',
                'holiday_not_sunday']
#[c for c in feature_whitelist if c in data.columns]
SCALE_COLS = [
    c for c in FEATURE_COLS
    if c in ["load_fc", "gen_fc_wind_on", "gen_fc_wind_off", "gen_fc_solar", "price_lag_24h", "price_lag_48h", "price_lag_168h"]
]

model_df = data[[TARGET_COL] + FEATURE_COLS].copy()
train_df = model_df.loc[(model_df.index >= TRAIN_START) & (model_df.index < TRAIN_END)].copy()
test_df = model_df.loc[(model_df.index >= TEST_START) & (model_df.index < TEST_END)].copy()
test_history_df = model_df.loc[
    (model_df.index >= TEST_START - pd.Timedelta(hours=LOOKBACK)) & (model_df.index < TEST_START)
].copy()
test_with_history_df = pd.concat([test_history_df, test_df], axis=0)

# helper functions for target transformation (for robustness against outliers)
c = np.median(np.abs(train_df[TARGET_COL]))
def asinh_transform(y, c):
    return np.arcsinh(y / c)
def asinh_inverse(y_t, c):
    return c * np.sinh(y_t)
# raw truth
y_test_raw = test_df[TARGET_COL].to_numpy(dtype=np.float32)


scaler = RobustScaler()
train_df.loc[:, SCALE_COLS] = scaler.fit_transform(train_df[SCALE_COLS])
train_df.loc[:, TARGET_COL] = asinh_transform(train_df[TARGET_COL], c)
test_df.loc[:, SCALE_COLS] = scaler.transform(test_df[SCALE_COLS])
test_df.loc[:, TARGET_COL] = asinh_transform(test_df[TARGET_COL], c)
test_history_df.loc[:, TARGET_COL] = asinh_transform(test_history_df[TARGET_COL].to_numpy(np.float32), c)
test_with_history_df = pd.concat([test_history_df, test_df], axis=0)
test_with_history_df.loc[test_df.index, SCALE_COLS] = test_df[SCALE_COLS]

### No-Lookback Setup

In [ ]:
def build_point_dataset(df: pd.DataFrame, feature_cols: list[str], target_col: str):
    X = df[feature_cols].to_numpy(dtype=np.float32)
    y = df[target_col].to_numpy(dtype=np.float32)
    ts = df.index
    return X, y, ts


X_train_point, y_train_point, ts_train_point = build_point_dataset(train_df, FEATURE_COLS, TARGET_COL)
X_test_point, y_test_point, ts_test_point = build_point_dataset(test_df, FEATURE_COLS, TARGET_COL)

pd.DataFrame([
    {"set": "point", "split": "train", "X": X_train_point.shape, "y": y_train_point.shape},
    {"set": "point", "split": "test", "X": X_test_point.shape, "y": y_test_point.shape},
])

,set,split,X,y
0,point,train,"(17544, 12)","(17544,)"
1,point,test,"(8760, 12)","(8760,)"


### Lookback Setup

In [ ]:
# Lookback setup: use the immediately preceding 168 hours for each target timestamp
SEQUENCE_COLS = [c for c in [TARGET_COL] if c in model_df.columns]


def build_lookback_dataset(
    df: pd.DataFrame,
    sequence_cols: list[str],
    exog_cols: list[str],
    target_col: str,
    lookback_hours: int,
):
    idx = df.index
    sequence_values = df[sequence_cols].to_numpy(dtype=np.float32)
    exog_values = df[exog_cols].to_numpy(dtype=np.float32)
    target_values = df[target_col].to_numpy(dtype=np.float32)

    X_seq, X_exog, y_target, ts_target = [], [], [], []

    for row_idx in range(lookback_hours, len(df)):
        keep_idx = np.arange(row_idx - lookback_hours, row_idx)
        seq_block = sequence_values[keep_idx, :]
        X_seq.append(seq_block)
        X_exog.append(exog_values[row_idx])
        y_target.append(target_values[row_idx])
        ts_target.append(idx[row_idx])

    return (
        np.stack(X_seq),
        np.stack(X_exog),
        np.asarray(y_target, dtype=np.float32),
        pd.DatetimeIndex(ts_target),
    )


# for LSTM
X_train_seq_lookback, X_train_exog_lookback, y_train_lookback, ts_train_lookback = build_lookback_dataset(
    train_df,
    SEQUENCE_COLS,
    FEATURE_COLS,
    TARGET_COL,
    LOOKBACK,
)

X_test_seq_lookback, X_test_exog_lookback, y_test_lookback, ts_test_lookback = build_lookback_dataset(
    test_with_history_df,
    SEQUENCE_COLS,
    FEATURE_COLS,
    TARGET_COL,
    LOOKBACK,
)
test_keep_mask = ts_test_lookback >= TEST_START
X_test_seq_lookback = X_test_seq_lookback[test_keep_mask]
X_test_exog_lookback = X_test_exog_lookback[test_keep_mask]
y_test_lookback = y_test_lookback[test_keep_mask]
ts_test_lookback = ts_test_lookback[test_keep_mask]

# for XGBoost
X_train_flat_lookback = np.concatenate([
    X_train_seq_lookback.reshape(X_train_seq_lookback.shape[0], -1),
    X_train_exog_lookback,
], axis=1)
X_test_flat_lookback = np.concatenate([
    X_test_seq_lookback.reshape(X_test_seq_lookback.shape[0], -1),
    X_test_exog_lookback,
], axis=1)

display(pd.DataFrame([
    {
        "set": "lookback",
        "split": "train",
        "X_seq": X_train_seq_lookback.shape,
        "X_exog": X_train_exog_lookback.shape,
        "X_flat": X_train_flat_lookback.shape,
        "y": y_train_lookback.shape,
    },
    {
        "set": "lookback",
        "split": "test",
        "X_seq": X_test_seq_lookback.shape,
        "X_exog": X_test_exog_lookback.shape,
        "X_flat": X_test_flat_lookback.shape,
        "y": y_test_lookback.shape,
    },
]))


rollout_df = pd.concat([
    model_df.loc[model_df.index < TEST_START, [TARGET_COL]],
    test_df[[TARGET_COL] + FEATURE_COLS],
], axis=0).sort_index()
rollout_df["_roll_price"] = rollout_df[TARGET_COL].astype(np.float32)
test_rollout_index = test_df.index
y_test_rollout = test_df[TARGET_COL].to_numpy(dtype=np.float32)


def get_rollout_inputs(
    rollout_df: pd.DataFrame,
    ts: pd.Timestamp,
    feature_cols: list[str],
    target_col: str,
    lookback_hours: int,
):
    hist_index = pd.date_range(
        end=ts - pd.Timedelta(hours=1),
        periods=lookback_hours,
        freq="h",
        tz=ts.tz,
    )
    hist_values = []
    for hist_ts in hist_index:
        source_col = target_col if hist_ts.normalize() < ts.normalize() else "_roll_price"
        hist_values.append(rollout_df.at[hist_ts, source_col])

    seq_block = np.asarray(hist_values, dtype=np.float32).reshape(lookback_hours, 1)
    exog_row = rollout_df.loc[ts, feature_cols].to_numpy(dtype=np.float32)

    return seq_block, exog_row


def rollout_predict_xgb(
    model,
    rollout_df: pd.DataFrame,
    target_times: pd.DatetimeIndex,
    feature_cols: list[str],
    target_col: str,
    lookback_hours: int,
):
    rollout_state = rollout_df.copy()
    preds = []

    for ts in target_times:
        seq_block, exog_row = get_rollout_inputs(rollout_state, ts, feature_cols, target_col, lookback_hours)
        x_row = np.concatenate([seq_block.reshape(-1), exog_row]).reshape(1, -1)
        y_hat = float(model.predict(x_row).reshape(-1)[0])

        rollout_state.at[ts, "_roll_price"] = y_hat
        preds.append(y_hat)

    return np.asarray(preds, dtype=np.float32)


def rollout_predict_lstm(
    model,
    rollout_df: pd.DataFrame,
    target_times: pd.DatetimeIndex,
    feature_cols: list[str],
    target_col: str,
    lookback_hours: int,
):
    rollout_state = rollout_df.copy()
    preds = []
    progress = 0
    total = len(target_times)

    for ts in target_times:
        progress += 1
        if progress % 500 == 0 or progress == total:
            print(f"LSTM rollout: {progress}/{total} - {ts}")

        seq_block, exog_row = get_rollout_inputs(rollout_state, ts, feature_cols, target_col, lookback_hours)
        #y_hat = float(model.predict([
        #    seq_block[np.newaxis, ...],
        #    exog_row.reshape(1, -1),
        #], verbose=0).reshape(-1)[0])
        y_hat = float(
            model(
                [seq_block[np.newaxis, ...], exog_row.reshape(1, -1)],
                training=False,
            ).numpy().reshape(-1)[0]
        )
        rollout_state.at[ts, "_roll_price"] = y_hat
        preds.append(y_hat)

    return np.asarray(preds, dtype=np.float32)



,set,split,X_seq,X_exog,X_flat,y
0,lookback,train,"(17376, 168, 1)","(17376, 12)","(17376, 180)","(17376,)"
1,lookback,test,"(8760, 168, 1)","(8760, 12)","(8760, 180)","(8760,)"


## ____________________ XGBoost ____________________




In [ ]:
xgb_base = dict(
    objective="reg:squarederror",
    random_state=1904,
    n_jobs=-1,
    device="cuda"
)

param_dist = {
    "n_estimators": [100, 200, 400, 500],
    "learning_rate": [0.01, 0.02, 0.03, 0.05, 0.08],
    "max_depth": [3, 4, 5, 6, 8],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "reg_lambda": [0.5, 1.0, 2.0, 5.0],
    "min_child_weight": [1, 3, 5, 7],
}

In [ ]:
# Point-Model tuning
tscv_point = TimeSeriesSplit(n_splits=5)

search_point = RandomizedSearchCV(
    estimator=XGBRegressor(**xgb_base),
    param_distributions=param_dist,
    n_iter=30,
    scoring="neg_mean_absolute_error",  # oder neg_root_mean_squared_error
    cv=tscv_point,
    verbose=1,
    random_state=1904,
    n_jobs=1,
)

search_point.fit(X_train_point, y_train_point)
xgb_point_model = search_point.best_estimator_
y_pred_xgb_point = xgb_point_model.predict(X_test_point)

print("Best params (point):", search_point.best_params_)
print("Best CV score (point):", -search_point.best_score_)


Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best params (point): {'subsample': 0.6, 'reg_lambda': 0.5, 'n_estimators': 500, 'min_child_weight': 3, 'max_depth': 6, 'learning_rate': 0.01, 'colsample_bytree': 0.6}
Best CV score (point): 0.13820775747299194


In [ ]:
# Lookback-Model tuning
tscv_lookback = TimeSeriesSplit(n_splits=5)

search_lookback = RandomizedSearchCV(
    estimator=XGBRegressor(**xgb_base),
    param_distributions=param_dist,
    n_iter=30,
    scoring="neg_mean_absolute_error",
    cv=tscv_lookback,
    verbose=1,
    random_state=1904,
    n_jobs=1,
)

search_lookback.fit(X_train_flat_lookback, y_train_lookback)
xgb_lookback_model = search_lookback.best_estimator_

y_pred_xgb_lookback = rollout_predict_xgb(
    xgb_lookback_model,
    rollout_df,
    test_rollout_index,
    FEATURE_COLS,
    TARGET_COL,
    LOOKBACK,
)

print("Best params (lookback):", search_lookback.best_params_)
print("Best CV score (lookback):", -search_lookback.best_score_)

Fitting 5 folds for each of 30 candidates, totalling 150 fits


Best params (lookback): {'subsample': 0.6, 'reg_lambda': 2.0, 'n_estimators': 500, 'min_child_weight': 5, 'max_depth': 6, 'learning_rate': 0.03, 'colsample_bytree': 1.0}
Best CV score (lookback): 0.05744587481021881


In [ ]:
# save models
joblib.dump(xgb_point_model, "xgb_point_model.joblib")
joblib.dump(xgb_lookback_model, "xgb_lookback_model.joblib")


['xgb_lookback_model.joblib']

In [ ]:
# load models
xgb_point = joblib.load("xgb_point_model.joblib")
xgb_lookback = joblib.load("xgb_lookback_model.joblib")

In [ ]:
y_pred_xgb_point = asinh_inverse(y_pred_xgb_point, c)
y_pred_xgb_lookback = asinh_inverse(y_pred_xgb_lookback, c)

In [ ]:
scores = pd.DataFrame([
    {"model": "xgb_a_point", "mae": mae_score(y_test_raw, y_pred_xgb_point), "rmse": rmse_score(y_test_raw, y_pred_xgb_point)},
    {"model": "xgb_b_rollout", "mae": mae_score(y_test_raw, y_pred_xgb_lookback), "rmse": rmse_score(y_test_raw, y_pred_xgb_lookback)},
]).sort_values("mae")

scores

,model,mae,rmse
0,xgb_a_point,15.9846,25.1600
1,xgb_b_rollout,17.9415,29.6435


In [ ]:
y_pred_xgb_point2_t = xgb_point.predict(X_test_point)
y_pred_xgb_point2 = asinh_inverse(y_pred_xgb_point2_t, c)

y_pred_xgb_lookback2_t = rollout_predict_xgb(
    xgb_lookback,
    rollout_df,
    test_rollout_index,
    FEATURE_COLS,
    TARGET_COL,
    LOOKBACK,
)
y_pred_xgb_lookback2 = asinh_inverse(y_pred_xgb_lookback2_t, c)


pd.DataFrame([
    {"model": "xgb_a_point", "mae": mae_score(y_test_raw, y_pred_xgb_point2), "rmse": rmse_score(y_test_raw, y_pred_xgb_point2)},
    {"model": "xgb_b_rollout", "mae": mae_score(y_test_raw, y_pred_xgb_lookback2), "rmse": rmse_score(y_test_raw, y_pred_xgb_lookback2)},
]).sort_values("mae")

/system/conda/miniconda3/envs/cloudspace/lib/python3.12/site-packages/xgboost/core.py:751: UserWarning: [10:11:35] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


,model,mae,rmse
0,xgb_a_point,15.9846,25.1600
1,xgb_b_rollout,17.9415,29.6435


In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# passend machen (falls unterschiedliche Längen)
y_true = np.asarray(y_test_raw).reshape(-1)   # oder y_test_raw, falls das dein echtes Ziel ist
y_pred = np.asarray(y_pred_xgb_point).reshape(-1)

n = min(len(y_true), len(y_pred), len(test_rollout_index))
df_plot = pd.DataFrame(
    {"Ist": y_true[:n], "XGB Lookback": y_pred[:n]},
    index=test_rollout_index[:n]
)

fig = go.Figure()
fig.add_trace(go.Scatter(x=df_plot.index, y=df_plot["Ist"], mode="lines", name="Ist"))
fig.add_trace(go.Scatter(x=df_plot.index, y=df_plot["XGB Lookback"], mode="lines", name="XGB Lookback"))

fig.update_layout(
    title="Ist vs. XGB Lookback (interaktiv zoombar)",
    xaxis_title="Zeit",
    yaxis_title="Preis",
    template="plotly_white",
    hovermode="x unified",
    height=700
)

# Range-Slider unten + Zoom-Buttons
fig.update_xaxes(
    rangeslider_visible=True,
    rangeselector=dict(
        buttons=[
            dict(count=1, label="1d", step="day", stepmode="backward"),
            dict(count=7, label="7d", step="day", stepmode="backward"),
            dict(count=1, label="1m", step="month", stepmode="backward"),
            dict(step="all", label="Alles"),
        ]
    )
)

fig.show()


In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# passend machen (falls unterschiedliche Längen)
y_true = np.asarray(y_test_raw).reshape(-1)   # oder y_test_raw, falls das dein echtes Ziel ist
y_pred = np.asarray(y_pred_xgb_lookback).reshape(-1)

n = min(len(y_true), len(y_pred), len(test_rollout_index))
df_plot = pd.DataFrame(
    {"Ist": y_true[:n], "XGB Lookback": y_pred[:n]},
    index=test_rollout_index[:n]
)

fig = go.Figure()
fig.add_trace(go.Scatter(x=df_plot.index, y=df_plot["Ist"], mode="lines", name="Ist"))
fig.add_trace(go.Scatter(x=df_plot.index, y=df_plot["XGB Lookback"], mode="lines", name="XGB Lookback"))

fig.update_layout(
    title="Ist vs. XGB Lookback (interaktiv zoombar)",
    xaxis_title="Zeit",
    yaxis_title="Preis",
    template="plotly_white",
    hovermode="x unified",
    height=700
)

# Range-Slider unten + Zoom-Buttons
fig.update_xaxes(
    rangeslider_visible=True,
    rangeselector=dict(
        buttons=[
            dict(count=1, label="1d", step="day", stepmode="backward"),
            dict(count=7, label="7d", step="day", stepmode="backward"),
            dict(count=1, label="1m", step="month", stepmode="backward"),
            dict(step="all", label="Alles"),
        ]
    )
)

fig.show()


In [ ]:
xgb_point_model = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=600,
    learning_rate=0.03,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=1904,
    n_jobs=-1,
)
xgb_point_model.fit(X_train_point, y_train_point)
y_pred_xgb_point_t = xgb_point_model.predict(X_test_point)
y_pred_xgb_point = mlog_inverse(y_pred_xgb_point_t, c)

xgb_lookback_model = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=600,
    learning_rate=0.03,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=1904,
    n_jobs=-1,
)
xgb_lookback_model.fit(X_train_flat_lookback, y_train_lookback)

y_pred_xgb_lookback_t = rollout_predict_xgb(
    xgb_lookback_model,
    rollout_df,
    test_rollout_index,
    FEATURE_COLS,
    TARGET_COL,
    LOOKBACK,
)
y_pred_xgb_lookback = mlog_inverse(y_pred_xgb_lookback_t, c)


## ____________________ LSTM ____________________





In [ ]:
# without rollout

y_pred_xgb_lookback2 = xgb_lookback.predict(X_test_flat_lookback)

y_pred_lstm_lookback2 = lstm_model.predict(
    [X_test_seq_lookback, X_test_exog_lookback],
    verbose=0,
).reshape(-1)

In [ ]:
input_seq = layers.Input(shape=(LOOKBACK, X_train_seq_lookback.shape[2]))
input_exog = layers.Input(shape=(X_train_exog_lookback.shape[1],))

hidden = layers.LSTM(64)(input_seq)
hidden = layers.Concatenate()([hidden, input_exog])
hidden = layers.Dense(64, activation="relu")(hidden)
hidden = layers.Dropout(0.2)(hidden)
output = layers.Dense(1)(hidden)

lstm_model = tf.keras.Model([input_seq, input_exog], output)
lstm_model.compile(optimizer="adam", loss="mae", metrics=[tf.keras.metrics.RootMeanSquaredError(name="rmse")])

early_stop = callbacks.EarlyStopping(patience=6, restore_best_weights=True)
_ = lstm_model.fit(
    [X_train_seq_lookback, X_train_exog_lookback],
    y_train_lookback,
    validation_split=0.1,
    epochs=40,
    batch_size=64,
    shuffle=False,
    callbacks=[early_stop],
    verbose=1,
)

y_pred_lstm_t = rollout_predict_lstm(
    lstm_model,
    rollout_df,
    test_rollout_index,
    FEATURE_COLS,
    TARGET_COL,
    LOOKBACK,a
)
y_pred_lstm = asinh_inverse(y_pred_lstm_t, c)


W0000 00:00:1774519988.364517   19461 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Epoch 1/40
245/245 ━━━━━━━━━━━━━━━━━━━━ 18s 66ms/step - loss: 0.2082 - rmse: 0.2867 - val_loss: 0.1660 - val_rmse: 0.3025
Epoch 2/40
245/245 ━━━━━━━━━━━━━━━━━━━━ 16s 67ms/step - loss: 0.1459 - rmse: 0.1913 - val_loss: 0.1433 - val_rmse: 0.2287
Epoch 3/40
245/245 ━━━━━━━━━━━━━━━━━━━━ 17s 68ms/step - loss: 0.1264 - rmse: 0.1654 - val_loss: 0.1322 - val_rmse: 0.1823
Epoch 4/40
245/245 ━━━━━━━━━━━━━━━━━━━━ 16s 66ms/step - loss: 0.1120 - rmse: 0.1474 - val_loss: 0.1059 - val_rmse: 0.1434
Epoch 5/40
245/245 ━━━━━━━━━━━━━━━━━━━━ 16s 65ms/step - loss: 0.1037 - rmse: 0.1359 - val_loss: 0.0960 - val_rmse: 0.1310
Epoch 6/40
245/245 ━━━━━━━━━━━━━━━━━━━━ 16s 65ms/step - loss: 0.0943 - rmse: 0.1258 - val_loss: 0.0887 - val_rmse: 0.1234
Epoch 7/40
245/245 ━━━━━━━━━━━━━━━━━━━━ 16s 65ms/step - loss: 0.0907 - rmse: 0.1213 - val_loss: 0.0932 - val_rmse: 0.1308
Epoch 8/40
245/245 ━━━━━━━━━━━━━━━━━━━━ 16s 66ms/step - loss: 0.0876 - rmse: 0.1177 - val_loss: 0.0852 - val_rmse: 0.1203
Epoch 9/40
245/245 ━━━━━

KeyboardInterrupt: 

## Comparison




In [ ]:
scores = pd.DataFrame([
    {"model": "xgb_a_point", "mae": mae_score(y_test_point, y_pred_xgb_point), "rmse": rmse_score(y_test_point, y_pred_xgb_point)},
    {"model": "xgb_b_rollout", "mae": mae_score(y_test_rollout, y_pred_xgb_lookback), "rmse": rmse_score(y_test_rollout, y_pred_xgb_lookback)},
    {"model": "xgb_c_lookback", "mae": mae_score(y_test_lookback, y_pred_xgb_lookback2), "rmse": rmse_score(y_test_lookback, y_pred_xgb_lookback2)},
    {"model": "lstm_rollout", "mae": mae_score(y_test_rollout, y_pred_lstm), "rmse": rmse_score(y_test_rollout, y_pred_lstm)},
    {"model": "lstm_lookback", "mae": mae_score(y_test_lookback, y_pred_lstm_lookback2), "rmse": rmse_score(y_test_lookback, y_pred_lstm_lookback2)}
]).sort_values("mae")

scores

,model,mae,rmse
2,xgb_c_lookback,7.4005,13.9812
4,lstm_lookback,9.9933,19.5538
1,xgb_b_rollout,20.2000,34.1966
0,xgb_a_point,22.8753,35.9608
3,lstm_rollout,30.8838,44.3106


In [ ]:
scores = pd.DataFrame([
    {"model": "xgb_a_point", "mae": mae_score(y_test_point, y_pred_xgb_point), "rmse": rmse_score(y_test_point, y_pred_xgb_point)},
    {"model": "xgb_b_rollout", "mae": mae_score(y_test_rollout, y_pred_xgb_lookback), "rmse": rmse_score(y_test_rollout, y_pred_xgb_lookback)},
]).sort_values("mae")

scores

,model,mae,rmse
0,xgb_a_point,16.2430,25.4623
1,xgb_b_rollout,18.9688,33.0888
